In [12]:
import pandas as pd
import numpy as np
import umap
import glob
import os
import joblib
from pathlib import Path

In [13]:
SAVEDIR = Path(os.path.expanduser("~/ECM_mod_output/UMAP/ODE_embeddings"))
REDDIR = Path(os.path.expanduser("~/ECM_mod_output/UMAP"))
if not os.path.exists(SAVEDIR):
    os.mkdir(SAVEDIR)

In [14]:
#if you have a lot of data for umaps

file_location = os.path.join('C:\\Users\\olesk\\Documents\\ode_runs_20230505\\ode_20230505\\ode_20230505\\', 'erl*.csv')
filenames = glob.glob(file_location)
f5 = filenames[0::5]

In [4]:
f5

['C:\\Users\\olesk\\Documents\\ode_runs_20230505\\ode_20230505\\ode_20230505\\erl_range_4-29_0001.csv',
 'C:\\Users\\olesk\\Documents\\ode_runs_20230505\\ode_20230505\\ode_20230505\\erl_range_4-29_0006.csv',
 'C:\\Users\\olesk\\Documents\\ode_runs_20230505\\ode_20230505\\ode_20230505\\erl_range_4-29_0011.csv',
 'C:\\Users\\olesk\\Documents\\ode_runs_20230505\\ode_20230505\\ode_20230505\\erl_range_4-29_0016.csv',
 'C:\\Users\\olesk\\Documents\\ode_runs_20230505\\ode_20230505\\ode_20230505\\erl_range_4-29_0021.csv',
 'C:\\Users\\olesk\\Documents\\ode_runs_20230505\\ode_20230505\\ode_20230505\\erl_range_4-29_0026.csv',
 'C:\\Users\\olesk\\Documents\\ode_runs_20230505\\ode_20230505\\ode_20230505\\erl_range_4-29_0031.csv',
 'C:\\Users\\olesk\\Documents\\ode_runs_20230505\\ode_20230505\\ode_20230505\\erl_range_4-29_0036.csv',
 'C:\\Users\\olesk\\Documents\\ode_runs_20230505\\ode_20230505\\ode_20230505\\erl_range_4-29_0041.csv',
 'C:\\Users\\olesk\\Documents\\ode_runs_20230505\\ode_20230505\\

In [15]:
#Important params to set the window size and step time distance
init_dt = 0.01                #initial dt from the raw model output
dt = 0.1                      #dt for umap window
window = 4                    #window size in hours
time_step = 0.2                 #time step between windows

#umap params
neighbs = 100
min_d = 0.05

#Load umap reducer object
reducer = joblib.load(str(REDDIR) + f'\\reducer_05_09_{init_dt}_{dt}_{window}_{time_step}.sav')

In [6]:
split_s = f.split('.')
split_ss = split_s[0].split('\\')
name = 'embedding_'+split_ss[-1]+'.csv'
name

NameError: name 'f' is not defined

In [7]:
test = np.arange(0,1000.01,0.01)
test

array([0.0000e+00, 1.0000e-02, 2.0000e-02, ..., 9.9998e+02, 9.9999e+02,
       1.0000e+03])

In [16]:
#Load and resample time based on umap window
#time_tmp = pd.read_csv('C:\\Users\\olesk\\ECM_mod_output\\ODE_runs\\exp_conditions\\exp_conditions\\ode_7-7_t.txt', sep=",", header=None)
#time_tmp = time_tmp.T
time_tmp = pd.DataFrame(data=np.arange(0,1000.01,0.01))
time_tmp.columns = ["time"]
time_tmp2 = time_tmp.iloc[::int(dt//init_dt)].reset_index().drop(columns=['index'])

In [19]:
#create window and total protein arrays
win_len = int(window/dt)
all_prot_tmp = np.empty((0,win_len*18))     #careful, this could end up being massive 
all_time_tmp = np.empty((0,win_len))
all_time_at_tmp = np.empty((0,1))
range_ind = np.arange(int(win_len/2),int(len(time_tmp2)-(int(win_len/2))),int(time_step/dt))

for f in filenames:
    prot_tmp = pd.read_csv(f, sep=",", header=None, usecols = range(0,18))
    prot_tmp.columns = ["skp2", "p21", "pp21", "cdh1", "myc", "cycD", "cycD-p21", 
    "Rb", "pRb", "E2F", "E2F-Rb", "cycE", "cycE-p21", "cycA", "cycA-p21",
    "cycB", "cycB-p21", "cdc20"]
    prot_tmp2 = prot_tmp.iloc[::int(dt//init_dt),:].reset_index().drop(columns=['index'])
    
    prot_re_tmp = np.empty((len(range_ind),win_len*18))
    time_re_tmp = np.empty((len(range_ind),win_len))
    time_at_tmp = np.empty((len(range_ind),1))

    time_tmp = pd.DataFrame(data=np.arange(0,len(prot_tmp)*0.01+0.01,0.01))
    time_tmp.columns = ["time"]
    time_tmp2 = time_tmp.iloc[::int(dt//init_dt)].reset_index().drop(columns=['index'])
    
    
    ii=0
    for n in range(int(win_len/2),int(len(time_tmp2)-(int(win_len/2))),int(time_step/dt)):
        tmp = prot_tmp2.iloc[n-int(win_len/2):n+int(win_len/2)]
        tmp_t = time_tmp2['time'].iloc[n-int(win_len/2):n+int(win_len/2)]
        tmp=tmp.to_numpy()
        tmp=np.reshape(np.transpose(tmp),win_len*18)
        tmp_at = time_tmp2['time'].iloc[n]
        
        prot_re_tmp[ii,:] = np.array([tmp])
        time_re_tmp[ii,:] = np.array([tmp_t])
        time_at_tmp[ii,:] = np.array([[tmp_at]])
        ii+=1
        
#     all_prot_tmp = np.append(all_prot_tmp, prot_re_tmp, axis=0)
#     all_time_tmp = np.append(all_time_tmp, time_re_tmp, axis=0)
#     all_time_at_tmp = np.append(all_time_at_tmp, time_at_tmp, axis=0)
    
    embedding_tmp = reducer.transform(prot_re_tmp)
    split_s = f.split('.')
    split_ss = split_s[0].split('\\')
    name = 'embedding_d_'+split_ss[-1]+'.csv'
    np.savetxt(str(SAVEDIR) + f'\\{name}', embedding_tmp, delimiter=',', fmt='%f')

KeyboardInterrupt: 